# Task 11: Hotel Cluster Recommendation

Решение по аналогии с `lab11.ipynb`: считаем взвешенные частоты кластеров по нескольким группировкам и собираем top-5 рекомендации для каждого тестового запроса.

In [1]:
import numpy as np
import pandas as pd

from collections import defaultdict
from pathlib import Path
from tqdm.auto import tqdm

/Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "task-11-data"

TRAIN_PATH = DATA_DIR / "train.csv"
DEST_PATH = DATA_DIR / "destinations.csv"
TEST_PATH = DATA_DIR / "test_files" / "Soc_Net_Task_5_Test_File_0.csv"
OUTPUT_PATH = BASE_DIR / "task-11-predictions.txt"
MAX_TRAIN_ROWS = None  # Например: 5_000_000 для быстрого чернового прогона.

for p in [TRAIN_PATH, DEST_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Not found: {p}")

print("Train:", TRAIN_PATH)
print("Destinations:", DEST_PATH)
print("Test:", TEST_PATH)

Train: /Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/task-11-data/train.csv
Destinations: /Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/task-11-data/destinations.csv
Test: /Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/task-11-data/test_files/Soc_Net_Task_5_Test_File_0.csv


In [3]:
def add_to_dict(d, key, cluster, score):
    d[key][int(cluster)] += float(score)


def top5_from_counter(counter):
    return [k for k, _ in sorted(counter.items(), key=lambda x: (-x[1], x[0]))[:5]]


def build_top5_map(d):
    return {k: top5_from_counter(v) for k, v in d.items()}


def merge_preds(*parts):
    result = []
    for part in parts:
        if part is None:
            continue
        for x in part:
            x = int(x)
            if x not in result:
                result.append(x)
            if len(result) == 5:
                return result
    return result


def safe_int(x):
    if pd.isna(x):
        return None
    return int(x)


def safe_float(x, ndigits=3):
    if pd.isna(x):
        return None
    return round(float(x), ndigits)

In [4]:
best_od_ulc = defaultdict(lambda: defaultdict(float))
best_search_dest = defaultdict(lambda: defaultdict(float))
best_market = defaultdict(lambda: defaultdict(float))
best_country = defaultdict(lambda: defaultdict(float))
best_dest = defaultdict(lambda: defaultdict(float))
popular = defaultdict(float)

usecols = [
    "date_time",
    "user_location_country",
    "user_location_region",
    "user_location_city",
    "orig_destination_distance",
    "srch_destination_id",
    "hotel_country",
    "hotel_market",
    "is_booking",
    "cnt",
    "hotel_cluster",
]

dtypes = {
    "user_location_country": "float32",
    "user_location_region": "float32",
    "user_location_city": "float32",
    "orig_destination_distance": "float32",
    "srch_destination_id": "int32",
    "hotel_country": "float32",
    "hotel_market": "float32",
    "is_booking": "int8",
    "cnt": "int16",
    "hotel_cluster": "int8",
}

reader = pd.read_csv(
    TRAIN_PATH,
    usecols=usecols,
    dtype=dtypes,
    chunksize=500_000,
    nrows=MAX_TRAIN_ROWS,
)

for chunk in tqdm(reader):
    chunk["date_time"] = pd.to_datetime(chunk["date_time"], errors="coerce")
    chunk["year"] = chunk["date_time"].dt.year.fillna(2013).astype(int)
    chunk["month"] = chunk["date_time"].dt.month.fillna(1).astype(int)
    chunk["orig_destination_distance"] = chunk["orig_destination_distance"].round(3)

    for row in chunk.itertuples(index=False):
        year = int(row.year)
        month = int(row.month)
        ulc = safe_int(row.user_location_country)
        ulr = safe_int(row.user_location_region)
        ulc_city = safe_int(row.user_location_city)
        odd = safe_float(row.orig_destination_distance, 3)
        sdi = int(row.srch_destination_id)
        hc = safe_int(row.hotel_country)
        hm = safe_int(row.hotel_market)
        is_booking = int(row.is_booking)
        cnt = int(row.cnt)
        cluster = int(row.hotel_cluster)

        time_weight = (year - 2012) * 12 + month
        base_score = time_weight * (1.0 + 4.0 * is_booking) * np.log1p(cnt)

        if ulc is not None and ulr is not None and ulc_city is not None and odd is not None:
            add_to_dict(best_od_ulc, (ulc, ulr, ulc_city, odd), cluster, base_score * 1.2)

        if sdi is not None and hc is not None and hm is not None:
            add_to_dict(best_search_dest, (sdi, hc, hm), cluster, base_score * 1.1)

        if hm is not None:
            add_to_dict(best_market, (hm,), cluster, base_score)

        if hc is not None:
            add_to_dict(best_country, (hc,), cluster, base_score * 0.8)

        add_to_dict(best_dest, (sdi,), cluster, base_score * 0.9)
        popular[cluster] += base_score

68it [02:57,  2.61s/it]


In [5]:
top_od_ulc = build_top5_map(best_od_ulc)
top_search_dest = build_top5_map(best_search_dest)
top_market = build_top5_map(best_market)
top_country = build_top5_map(best_country)
top_dest = build_top5_map(best_dest)
top_popular = top5_from_counter(popular)

print("Popular fallback:", top_popular)

Popular fallback: [91, 48, 41, 64, 65]


In [6]:
test_df = pd.read_csv(TEST_PATH)
print("Test rows:", len(test_df))

predictions = []

for row in tqdm(test_df.itertuples(index=False), total=len(test_df)):
    ulc = safe_int(getattr(row, "user_location_country"))
    ulr = safe_int(getattr(row, "user_location_region"))
    ulc_city = safe_int(getattr(row, "user_location_city"))
    odd = safe_float(getattr(row, "orig_destination_distance"), 3)
    sdi = safe_int(getattr(row, "srch_destination_id"))
    hc = safe_int(getattr(row, "hotel_country"))
    hm = safe_int(getattr(row, "hotel_market"))

    p1 = None
    if ulc is not None and ulr is not None and ulc_city is not None and odd is not None:
        p1 = top_od_ulc.get((ulc, ulr, ulc_city, odd))

    p2 = None
    if sdi is not None and hc is not None and hm is not None:
        p2 = top_search_dest.get((sdi, hc, hm))

    p3 = None
    if hm is not None:
        p3 = top_market.get((hm,))

    p4 = None
    if hc is not None:
        p4 = top_country.get((hc,))

    p5 = None
    if sdi is not None:
        p5 = top_dest.get((sdi,))

    pred = merge_preds(p1, p2, p5, p3, p4, top_popular)
    predictions.append(pred)

assert len(predictions) == 10_000, f"Expected 10000 predictions, got {len(predictions)}"
assert all(1 <= len(p) <= 5 for p in predictions)

answer = str(predictions).replace(" ", "")
OUTPUT_PATH.write_text(answer, encoding="utf-8")

print("Saved:", OUTPUT_PATH)
print("First 3 predictions:", predictions[:3])

Test rows: 10000


100%|██████████| 10000/10000 [00:00<00:00, 211216.95it/s]

Saved: /Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/task-11-predictions.txt
First 3 predictions: [[89, 53, 26, 84, 73], [82, 28, 36, 12, 85], [97, 25, 90, 64, 58]]


In [7]:
answer[:300]

'[[89,53,26,84,73],[82,28,36,12,85],[97,25,90,64,58],[63,92,58,62,57],[47,98,70,41,56],[26,73,84,92,0],[95,98,21,68,72],[1,45,79,24,54],[36,62,46,85,67],[46,36,62,58,9],[55,9,21,95,68],[71,34,77,0,54],[65,31,52,87,64],[62,29,12,36,85],[40,17,83,30,23],[64,46,99,25,11],[25,59,0,31,96],[68,9,91,28,59],'